## Splitting Q, K, V

- In the previous section we produced ``(B, T, 3C)``
- We now cut the last dimension into three equal continguous chunks of size C producing three seperate tensors.
- The function used is ``torch.split()`` this divides the tensor into chunks along a chosen dimension. 
- ``torch.split(C, dim=2)`` where:
  - ``C`` - the size of each chunk
  - ``dim=2`` - which dimension to split so indices (0 = `B`, 1 = `T`, 2 = `3C`)
- This split will leave `B` and `T` intact. 
- **Alternatively** use the ``dim=-1`` since this minimizes the assumption on the dimension of the tensor
   (only assumes that the embedded token vectors are always loacted in the last index)

**Good to know** 

Another way to implement the split is as follows:

```python
q = qkv[:, :, :C]
k = qkv[:, :, C:2*C]
v = qkv[:, :, 2*C:]
```
Where I translate ``:``  as "keep in this dimension index all values ``from and until (including)``"

## Reshape into Heads

- We now have `Q`, `K`, `V` in the shape `(B, T, C)`. Now Multi-head attention splits each into `h` **head** where each head works with a `C/h-dimension` slice. The target shape for each is `(B, h, T, C/h)`.

To do this we:
1. `rehsape` from `(B, T, C)` to `(B, T, h, C/h)`
2. `transpose` from `(B, T, h, C/h)` to `(B, h, T, C/h)`

Recall what (B, T, C) actually means: `(B number of prompts, containing T tokens per prompt, where each token is better explained as a vector of size C)`

by converting to `(B, T, h, C/h)` we have a head actually focusing on a specific subspace of the token, which enables attention capability to some degree, however, with th current shape it's only going to focus at a single token and not all tokens, thus by grouping `(B, h, T, C/h)`, attention focuses on a specific subspace but across **all** tokens.

- The reshaping is done by ``.view()`` by providing the dimension you wish to `view` the tensor as, without moving data so long as the product of the dimension is the same.
- The transpose is done by `.transpose(,)` which two indices you wish to swap

### Implementation

In [8]:
import torch 
import torch.nn as nn

In [2]:
class MultiHeadQKV(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3*n_embd)

    def split_heads(self, t):
        B, T, C = t.shape
        h = self.n_head
        reshaped = t.view(B, h, T, C//h)
        return reshaped.transpose(1,2)

    def forward(self, x):
        q, k, v = self.qkv(x).split(self.n_embd, dim=-1)
        q_split = self.split_heads(q)
        k_split = self.split_heads(k)
        v_split = self.split_heads(v)
        return q_split, k_split , v_split 

        

In [3]:
m = MultiHeadQKV(n_embd=4, n_head=2)
x = torch.tensor([[[1., 2., 3., 4.],
                   [5., 6., 7., 8.]]])
q, k, v = m(x)
print("Query", q)
print("Key", k)
print("Value", v)

Query tensor([[[[ 3.3982,  1.0682],
          [ 7.3231,  4.7888]],

         [[-1.1091, -2.1500],
          [-2.9963, -3.9680]]]], grad_fn=<TransposeBackward0>)
Key tensor([[[[-1.6056, -1.2618],
          [-4.5765, -3.2542]],

         [[ 0.6545, -0.8049],
          [ 2.1640, -3.5888]]]], grad_fn=<TransposeBackward0>)
Value tensor([[[[-1.0861, -1.9944],
          [-1.2283, -6.1566]],

         [[ 0.0918,  0.5130],
          [ 0.3521,  0.3163]]]], grad_fn=<TransposeBackward0>)


In [9]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, n_embed, block_size, n_head):
        super().__init__()
        self.embed = n_embed
        self.n_head = n_head
        self.wte = nn.Embedding(vocab_size, n_embed)
        self.wpe = nn.Embedding(block_size, n_embed)
        self.ln = nn.LayerNorm(n_embed)
        self.qkv = nn.Linear(n_embed, 3*n_embed)


    def final_input(self, x):
        token_x = self.wte(x)
        B, T = x.shape
        pos = torch.arange(T)
        pos_x = self.wpe(pos)
        x_meged = token_x + pos_x
        x_norm = self.ln(x_meged)
        return x_norm

    def reshape_head(self, t):
        # (B, T, C) -> (B, T, h, C/h) -> (B, h, T, C/h)
        B, T, C = t.shape
        h = self.n_head
        new_view = t.view(B, T, h, C//h)
        return new_view.transpose(B, h, T, C//h)


    def forward(self, idx):
        x = self.final_input(idx)
        expanded_x = self.qkv(x)
        q_raw, k_raw, v_raw = expanded_x.split(self.embed, dim=-1)
        q = self.reshape(q_raw)
        k = self.reshape(k_raw)
        v = self.reshape(v_raw)
        return q, k, v



In [10]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, block_size, n_embed, n_heads):
        super().__init__()
        self.n_embed = n_embed
        self.n_heads = n_heads
        self.wte = nn.Embedding(vocab_size, n_embed)
        self.wpe = nn.Embedding(block_size, n_embed)
        self.ln = nn.LayerNorm(normalized_shape=n_embed)
        self.qkv = nn.Linear(n_embed, 3*n_embed)

    def produce_input(self, t):
        emb_token = self.wte(t)
        B, T = t.shape
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        x_unnormalised = emb_token + pos_emb
        x_normalised = self.ln(x_unnormalised)
        return x_normalised

    def reshape_heads(self, x):
        B, T, C = x.shape
        h = self.n_heads
        x_reviewed = x.view(B, T, h, C//h)
        x_reshaped = x_reviewed.transpose(1, 2)
        return x_reshaped
        

    def forward(self, prompt_in):
        x = self.produce_input(prompt_in)
        qkv = self.qkv(x)
        q_raw, k_raw, v_raw = qkv.split(self.embed, dim=-1)
        q = self.reshape_heads(q_raw)
        k = self.reshape_heads(k_raw)
        v = self.reshape_heads(v_raw)
        return q, k, v

In [13]:
t = Transformer(vocab_size=1, block_size=4, n_embed=4, n_heads=2)
x = torch.tensor([[[1., 2., 3., 4.],
                   [5., 6., 7., 8.]]])   # (B, T, C) = (1, 2, 4)
q, k, v = m(x)
print(q.shape)   # torch.Size([1, 2, 2, 2]) = (B, h, T, C/h)

torch.Size([1, 2, 2, 2])
